## Tokens and Tokenization

A token is the actual unit of text an LLM processes — not a word, not a character, but a piece somewhere in between, decided by an algorithm before the model ever sees the text.
Every cost calculation, every context window limit, every rate limit you have worked with is measured in tokens, not words or characters.

## How it actually works

**Tokenization happens BEFORE the model sees anything:**
When you send text to an LLM, a separate process called a tokenizer breaks that text
into tokens first. The model never actually "sees" raw letters or words — it only
ever sees a sequence of token IDs (numbers), each one referring to an entry in a
fixed vocabulary the tokenizer was built with.

**Why tokens are not the same as words:**
Common words often become a single token. Less common words, unusual names, or
words in less-represented languages often get split into multiple tokens.
A rough rule of thumb for English: 1 token is approximately 4 characters,
or about 0.75 words — but this varies significantly depending on the specific text.

**Example of the mismatch (illustrated properly in the code cell below):**
"ERP" might be one token. "supercalifragilisticexpialidocious" might be split into
five or six smaller pieces, because that specific sequence of characters was rare
enough in training data that the tokenizer never assigned it a single dedicated token.

**Why this matters for cost specifically:**
API pricing (covered in detail in a later notebook) is charged per token, separately
for input tokens and output tokens. The same sentence can cost more or less depending
on word choice, language, and even formatting — not because of meaning, but because
of how many tokens the specific characters happen to break into.

**Why this matters for context window limits:**
A 200,000 token context window is not 200,000 words — it is typically closer to
150,000 words of English text, though this varies. Code, unusual formatting, and
non-English text often consume MORE tokens per "unit of meaning" than plain English,
meaning the same conceptual amount of content can fill the context window faster
than people often expect.

## Where it breaks or gets dangerous

**Silent cost surprises from unexpected tokenization:**
Code, especially with unusual variable names, symbols, and indentation, frequently
tokenizes LESS efficiently than plain prose — meaning the same amount of "stuff happening"
in a request can cost meaningfully more tokens than expected if you are not accounting
for this. A prompt full of X++ code or JSON structures will typically consume more
tokens than the same character count of plain English.

**Truncation surprises — cutting off mid-token, not mid-character:**
If you are manually trying to estimate how much text fits in a token budget by
counting characters, you can be significantly off, because the actual cutoff happens
at a TOKEN boundary, not a character boundary. This matters when building any system
that manually manages context window budget (your RAG Assistant, for instance) —
rough character-based estimates can lead to either wasting available context space
or accidentally exceeding the limit.

**Different languages tokenize very differently:**
English generally tokenizes efficiently because the vast majority of training data
the tokenizer was built from was English. Other languages — especially those with
different writing systems — can require significantly MORE tokens to represent the
same amount of meaning. This has real cost and context-window implications for any
multilingual application, and is worth knowing given Singapore's multilingual context.

**Tokenizers differ between model providers:**
A token in Claude's tokenizer is not necessarily the same as a token in another
provider's tokenizer. If you are ever comparing costs or context limits across
different providers, you cannot assume token counts are directly comparable —
the same text can tokenize to a different number of tokens depending on which
tokenizer is doing the counting.

In [ ]:
# Tokens and Tokenization — illustrating how text breaks into tokens differently

import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def count_tokens_for_text(text, label):
    """Use Claude's token counting endpoint to see actual token usage."""
    response = client.messages.count_tokens(
        model="claude-sonnet-4-6",
        messages=[{"role": "user", "content": text}]
    )
    char_count = len(text)
    token_count = response.input_tokens
    ratio = char_count / token_count if token_count > 0 else 0
    print(f"{label}")
    print(f"  Text: {text[:60]}{'...' if len(text) > 60 else ''}")
    print(f"  Characters: {char_count}")
    print(f"  Tokens:     {token_count}")
    print(f"  Chars per token: {ratio:.2f}")
    print()

# Plain English sentence
count_tokens_for_text(
    "The three-way matching rule compares purchase order, receipt, and invoice values.",
    "=== Plain English sentence ==="
)

# Same length, but technical/code-like content
count_tokens_for_text(
    "{'PO_QTY': 100, 'RCPT_QTY': 100, 'INV_QTY': 98, 'TOLERANCE_PCT': 0.02}",
    "=== JSON-style structured text (same approximate length) ==="
)

# A single uncommon word
count_tokens_for_text(
    "supercalifragilisticexpialidocious",
    "=== A single long, uncommon English word ==="
)

# A single common word for comparison
count_tokens_for_text(
    "company",
    "=== A single short, common English word ==="
)

print("Notice: the JSON-style text likely used MORE tokens than the plain English")
print("sentence of similar character length, due to symbols and structure.")
print("The uncommon long word likely split into multiple tokens,")
print("while the common short word likely became a single token.")

## My D365 analogy

Tokens are like financial dimension combinations in D365 — the actual unit the system
processes is not "an account" in the casual sense you think about it, it is the specific
combination of main account plus every active financial dimension attached to it.

Two transactions that look conceptually similar to a human — "marketing expense"
and "marketing expense, different cost centre" — can be entirely different combinations
to the system underneath, consuming separate space in whatever structure tracks them,
even though a person glancing at both would call them "basically the same thing."

Just like estimating report complexity by counting visible account names instead of
actual dimension combinations gives you a wrong picture of what the system is really
processing, estimating LLM cost or context usage by counting words instead of tokens
gives you a wrong picture of what is actually being charged and consumed underneath.

The lesson is the same in both cases: the unit the SYSTEM actually counts is not always
the unit a human would naturally count by eye, and assuming they match leads to
budget and capacity surprises.

## LinkedIn post idea

A 200,000 token context window sounds enormous, until you realise a token is not a word.

Tokens are the actual unit a language model processes — pieces of text decided by an
algorithm before the model ever sees your input. Common words are usually one token.
Code, symbols, and uncommon words often split into several.

This matters more than it sounds. The same conceptual amount of content — a paragraph
of plain English versus the equivalent in JSON or code — can consume noticeably
different amounts of your token budget, even though a human would call them roughly
equal in "how much stuff is in there."

Every cost figure, every context limit, every rate limit in this field is measured
in tokens, not words. Estimating any of those by counting words instead is a bit like
estimating a D365 report's complexity by counting visible account names instead of
actual financial dimension combinations.

The unit the system actually counts is rarely the unit a human counts by eye.
That gap is where budget surprises live, in ERP systems and in AI systems alike.